In [ ]:
import os

from googleapiclient.discovery import build
import pandas as pd
from tqdm import tqdm

API_KEY = os.environ["YOUTUBE_API_KEY"]  # 본인 API 키
youtube = build("youtube", "v3", developerKey=API_KEY)

df = pd.read_csv("liverpool_youtube_comments_clean_감정분석_v2.csv")

# 고유 video_id만 추출
video_ids = df["video_id"].dropna().unique().tolist()

video_info = {}

for v_id in tqdm(video_ids):
    request = youtube.videos().list(
        part="snippet",
        id=v_id
    )
    response = request.execute()

    if response["items"]:
        snippet = response["items"][0]["snippet"]
        video_info[v_id] = {
            "video_title": snippet["title"],
            "video_published_at": snippet["publishedAt"],
            "channel_title": snippet["channelTitle"]
        }
    else:
        video_info[v_id] = {
            "video_title": "영상 정보 없음",
            "video_published_at": None,
            "channel_title": None
        }

# video_id 기준으로 제목 붙이기
df["video_title"] = df["video_id"].map(lambda x: video_info.get(x, {}).get("video_title"))
df["video_published_at"] = df["video_id"].map(lambda x: video_info.get(x, {}).get("video_published_at"))
df["channel_title"] = df["video_id"].map(lambda x: video_info.get(x, {}).get("channel_title"))

display(df[["video_id", "video_title", "video_published_at", "channel_title"]].drop_duplicates())

df.to_csv("liverpool_gpt_all_labeled_with_title.csv", index=False, encoding="utf-8-sig")
df.to_excel("liverpool_gpt_all_labeled_with_title.xlsx", index=False)

print("✅ 영상 제목 추가 완료")

In [ ]:
import pandas as pd
import numpy as np
import re

# =====================================================
# 0. 파일 경로 설정
# =====================================================

YOUTUBE_FILE = "liverpool_gpt_all_labeled_with_title.xlsx"
CAFE_FILE = "liverpool_cafe_38R_before_after.xlsx"

OUTPUT_TOTAL_CSV = "liverpool_youtube_cafe_concat.csv"
OUTPUT_TOTAL_XLSX = "liverpool_youtube_cafe_concat.xlsx"

OUTPUT_REANALYZE_CSV = "liverpool_reanalyze_target.csv"
OUTPUT_REANALYZE_XLSX = "liverpool_reanalyze_target.xlsx"


# =====================================================
# 1. 기준일 설정
# =====================================================

# 38R 기준일
MATCH_38R = pd.Timestamp("2026-05-25 00:00:00")


# =====================================================
# 2. 함수 정의
# =====================================================

def extract_round_from_title(title):
    """
    영상 제목에서 29R, 30R, 38R 같은 라운드 정보 추출
    예: "[프리미어리그] 34R 리버풀 vs ..." -> 34R
    """
    title = str(title)

    match = re.search(r"(\d{1,2})R", title)

    if match:
        return f"{match.group(1)}R"

    return None


def get_period_by_date(dt, 기준일=MATCH_38R):
    """
    38R 기준 전/후 구분
    """
    if pd.isna(dt):
        return None

    if dt < 기준일:
        return "before_38R"
    else:
        return "after_38R"


def to_datetime_no_timezone(series):
    """
    Excel 저장 오류 방지를 위해 timezone 제거
    """
    return pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert(None)


# =====================================================
# 3. 파일 불러오기
# =====================================================

youtube_df = pd.read_excel(YOUTUBE_FILE)
cafe_df = pd.read_excel(CAFE_FILE)

print("유튜브 원본 shape:", youtube_df.shape)
print("카페 원본 shape:", cafe_df.shape)

print("\n유튜브 컬럼:")
print(youtube_df.columns.tolist())

print("\n카페 컬럼:")
print(cafe_df.columns.tolist())


# =====================================================
# 4. 유튜브 데이터 공통 포맷으로 변환
# =====================================================

youtube_final = pd.DataFrame(index=youtube_df.index)

youtube_final["source"] = "youtube"
youtube_final["round"] = youtube_df["video_title"].apply(extract_round_from_title)
youtube_final["club"] = "Tottenham"
youtube_final["title"] = youtube_df["video_title"]
youtube_final["content_type"] = "video_comment"

youtube_final["text_original"] = youtube_df["comment_original"]
youtube_final["text_clean"] = youtube_df["comment_clean"]

youtube_final["content_published_at"] = to_datetime_no_timezone(
    youtube_df["video_published_at"]
)

youtube_final["comment_published_at"] = to_datetime_no_timezone(
    youtube_df["published_at"]
)

youtube_final["like_count"] = youtube_df["like_count"]
youtube_final["url"] = youtube_df["video_url"]

youtube_final["period"] = youtube_final["comment_published_at"].apply(
    get_period_by_date
)

youtube_final["gpt_sentiment"] = youtube_df["gpt_sentiment"]
youtube_final["gpt_confidence"] = youtube_df["gpt_confidence"]
youtube_final["gpt_target"] = youtube_df["gpt_target"]
youtube_final["gpt_reason"] = youtube_df["gpt_reason"]

youtube_final["topic"] = None


# =====================================================
# 5. 카페 데이터 공통 포맷으로 변환
# =====================================================

cafe_final = pd.DataFrame(index=cafe_df.index)

cafe_final["source"] = "cafe"
cafe_final["round"] = cafe_df["round"]
cafe_final["club"] = "Tottenham"
cafe_final["title"] = cafe_df["title"]

cafe_final["content_type"] = cafe_df["content_type"].map({
    "post": "cafe_post",
    "comment": "cafe_comment",
    "cafe_post": "cafe_post",
    "cafe_comment": "cafe_comment"
})

cafe_final["text_original"] = cafe_df["text_original"]
cafe_final["text_clean"] = cafe_df["text_clean"]

cafe_final["content_published_at"] = pd.to_datetime(
    cafe_df["post_date"],
    errors="coerce"
)

cafe_final["comment_published_at"] = pd.to_datetime(
    cafe_df["comment_date"],
    errors="coerce"
)

cafe_final["like_count"] = None
cafe_final["url"] = cafe_df["url"]
cafe_final["period"] = cafe_df["period"]

cafe_final["gpt_sentiment"] = None
cafe_final["gpt_confidence"] = None
cafe_final["gpt_target"] = None
cafe_final["gpt_reason"] = None
cafe_final["topic"] = None


# =====================================================
# 6. 최종 컬럼 순서 통일
# =====================================================

final_columns = [
    "source",
    "round",
    "club",
    "title",
    "content_type",
    "text_original",
    "text_clean",
    "content_published_at",
    "comment_published_at",
    "like_count",
    "url",
    "period",
    "gpt_sentiment",
    "gpt_confidence",
    "gpt_target",
    "gpt_reason",
    "topic"
]

youtube_final = youtube_final[final_columns]
cafe_final = cafe_final[final_columns]


# =====================================================
# 7. concat
# =====================================================

total_df = pd.concat(
    [youtube_final, cafe_final],
    ignore_index=True
)

# 텍스트 없는 행 제거
total_df = total_df[
    total_df["text_clean"].notna() &
    (total_df["text_clean"].astype(str).str.len() >= 2)
].reset_index(drop=True)


# =====================================================
# 8. 재분석 대상 추출
#    - 유튜브에서 기존 gpt_sentiment가 neutral/중립인 데이터
#    - 카페처럼 gpt_sentiment가 None인 데이터
# =====================================================

reanalyze_df = total_df[
    total_df["gpt_sentiment"].isna() |
    total_df["gpt_sentiment"].isin(["neutral", "중립"])
].copy()


# =====================================================
# 9. 확인
# =====================================================

print("\n==============================")
print("데이터 변환 결과")
print("==============================")

print("youtube_final:", youtube_final.shape)
print("cafe_final:", cafe_final.shape)
print("total_df:", total_df.shape)
print("reanalyze_df:", reanalyze_df.shape)

print("\n[source 분포]")
print(total_df["source"].value_counts(dropna=False))

print("\n[round 분포]")
print(total_df["round"].value_counts(dropna=False).sort_index())

print("\n[period 분포]")
print(total_df["period"].value_counts(dropna=False))

print("\n[content_type 분포]")
print(total_df["content_type"].value_counts(dropna=False))

print("\n[gpt_sentiment 분포]")
print(total_df["gpt_sentiment"].value_counts(dropna=False))

display(total_df.head())

display(
    reanalyze_df[
        [
            "source",
            "round",
            "period",
            "title",
            "content_type",
            "text_clean",
            "gpt_sentiment",
            "gpt_target"
        ]
    ].head()
)


# =====================================================
# 10. 저장
# =====================================================

# CSV 저장
total_df.to_csv(
    OUTPUT_TOTAL_CSV,
    index=False,
    encoding="utf-8-sig"
)

reanalyze_df.to_csv(
    OUTPUT_REANALYZE_CSV,
    index=False,
    encoding="utf-8-sig"
)

# Excel 저장용 복사본
total_df_excel = total_df.copy()
reanalyze_df_excel = reanalyze_df.copy()

# Excel은 timezone 있는 datetime을 저장 못하므로 다시 한번 제거
date_cols = [
    "content_published_at",
    "comment_published_at"
]

for col in date_cols:
    total_df_excel[col] = pd.to_datetime(
        total_df_excel[col],
        errors="coerce",
        utc=True
    ).dt.tz_convert(None)

    reanalyze_df_excel[col] = pd.to_datetime(
        reanalyze_df_excel[col],
        errors="coerce",
        utc=True
    ).dt.tz_convert(None)

# Excel 저장
total_df_excel.to_excel(
    OUTPUT_TOTAL_XLSX,
    index=False
)

reanalyze_df_excel.to_excel(
    OUTPUT_REANALYZE_XLSX,
    index=False
)

print("\n✅ 저장 완료")
print("전체 통합 CSV:", OUTPUT_TOTAL_CSV)
print("전체 통합 Excel:", OUTPUT_TOTAL_XLSX)
print("재분석 대상 CSV:", OUTPUT_REANALYZE_CSV)
print("재분석 대상 Excel:", OUTPUT_REANALYZE_XLSX)